In [1]:
import pathlib
import pickle

import folium
import mcr_py.helper_functions
import mcr_py.mcr.data
import mcr_py.mcr.path
import mcr_py.mcr5.labels
import mcr_py.minute_city.minute_city
import mcr_py.utils.strtime
import numpy as np
import pandas as pd
import polars as pl
from mcr_py.mcr.path import GTFSPath, Path, PathType
from mcr_py.utils.logger import setup

setup("INFO")

In [2]:
city_name = "cologne"
date = "20250926"

In [3]:
data_directory = pathlib.Path("../data/")
base_directory = data_directory / date
cache_path = base_directory / "cache/"
osm_path = base_directory / "osm_raw"
geometa_path = base_directory / f"cache/{city_name}_geometa.json"
mcr5_output_path = base_directory / f"mcr5_results/{city_name}_reduced_paths"
mcr5_output_path_comp = base_directory / f"mcr5_results/{city_name}"
gtfs_clean_dir = base_directory / f"gtfs_clean/{city_name}/"
gtfs_clean_struct = gtfs_clean_dir / "structs.pkl"
gtfs_clean_stops = gtfs_clean_dir / "stops.parquet"
geo_meta, geo_data = mcr_py.helper_functions.load_auxiliary_classes(
    geo_meta_path=geometa_path,
    city_id="Koeln",
    osm_path=osm_path,
    cache_path=cache_path,
)

[10:56:25] INFO     Loading OSM walking                               ]8;id=641149;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=411861;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#63\63]8;;\
[10:56:28] INFO     Loading OSM walking done (2.76 seconds)           ]8;id=684277;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=78234;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#63\63]8;;\
           INFO     Loading OSM POIs                                  ]8;id=870209;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=147107;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#70\70]8;;\
           INFO     Loading OSM POIs done (0.01 seconds)              ]8;id=203848;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=3719;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#70\70]8;;\
           INFO     Loading

In [4]:
with open(mcr5_output_path / "bicycle" / "891fa199c77ffff.pkl", "rb") as f:
    hex = pickle.load(f)

In [5]:
comp = pl.read_ipc(mcr5_output_path_comp / "bicycle" / "891fa199c77ffff.feather").join(
    geo_data.pois.select("nearest_osm_node", "poi_type", "lat", "long"),
    how="left",
    left_on="osm_node_id",
    right_on="nearest_osm_node",
)
comp.head()

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


osm_node_id,time,cost,n_transfers,human_readable_time,poi_type,lat,long
i64,i64,i64,i64,str,str,f64,f64
4820041772,28953,0,0,"""08:02:33""",null,null,null
492164783,29060,0,0,"""08:04:20""",null,null,null
4820041792,29113,0,0,"""08:05:13""",null,null,null
1895145117,29108,0,0,"""08:05:08""",null,null,null
4820041814,28965,0,0,"""08:02:45""",null,null,null


In [6]:
labels = pd.DataFrame(
    [
        (label.node_id, label.values[0], label.values[1], n_transfers, label)
        for n_transfers, bags in hex["bags_i"].items()
        for bag in bags.values()
        for label in bag
    ],
    columns=["osm_node_id", "time", "cost", "n_transfers", "label"],
)

In [7]:
labels = labels.merge(
    geo_data.pois.select(
        pl.col("nearest_osm_node").alias("osm_node_id").cast(pl.Int64),
        pl.col("lat").alias("poi_lat"),
        pl.col("long").alias("poi_long"),
        "poi_type",
    ).to_pandas(),
    how="left",
    on="osm_node_id",
)

In [8]:
nodes = geo_data.osm_nodes.with_columns(pl.col("osm_id").alias("id")).to_pandas()

In [9]:
path_manager = hex["path_manager"]

In [10]:
from mcr_py.mcr.data import NetworkType

translator_map = {
    PathType.WALKING: dict(
        geo_data.osm_nodes.select(pl.col("rx_node_id").alias("osm"), "osm_id").rows()
    ),
    PathType.CYCLING_WALKING: dict(
        pl.concat(
            [
                geo_data.osm_nodes.select(
                    pl.lit("W").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
                geo_data.additional_networks[NetworkType.CYCLING][0].select(
                    pl.lit("D").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
            ],
            how="diagonal",
        )
        .with_row_index()
        .rows()
    ),
    # PathType.DRIVING_WALKING: reverse_node_map,
    PathType.PUBLIC_TRANSPORT: None,
}

In [11]:
def format_meta(meta, previous_meta, start_time):
    values = meta["values"]
    arrival_time = values[0]
    cost = values[1]

    if previous_meta:
        previous_values = previous_meta["values"]
        previous_arrival_time = previous_values[0]
        previous_cost = previous_values[1]

        arrival_time -= previous_arrival_time
        cost -= previous_cost
    else:
        arrival_time -= start_time

    return f"{mcr_py.utils.strtime.seconds_to_str_time(arrival_time, 10)} ({cost})"

In [13]:
labels.poi_type.unique()

array([nan, 'Shops', 'Health', 'Sustenance', 'Grocery', 'Banks',
       'Education', 'Parks'], dtype=object)

In [14]:
color_map = {
    "Shops": "orange",
    "Grocery": "red",
    "Parks": "green",
    "Education": "blue",
    "Banks": "purple",
    "Health": "darkgreen",
    "Sustenance": "yellow",
}

In [ ]:
from mcr_py.mcr.label import IntermediateLabel

toloop = labels

# stops_by_id = stops_df.set_index("stop_id")
sample_label = labels.iloc[0]
sample_node_id = sample_label.osm_node_id
nodes_by_id = nodes.set_index("id", drop=False)
sample_node = nodes_by_id.loc[sample_node_id]
start_time = 288000

m = folium.Map(location=[sample_node.lat, sample_node.long], zoom_start=13)


for row in toloop.itertuples():
    label: IntermediateLabel = row.label  # pyright: ignore[reportAssignmentType]
    end_node_id = row.osm_node_id
    end_node = nodes_by_id.loc[end_node_id]

    folium.CircleMarker(
        location=[end_node.lat, end_node.long],  # pyright: ignore[reportArgumentType]
        popup=f"End: {end_node_id}",
        color="black",
        radius=1,
    ).add_to(m)
    if row.poi_type is not np.nan:
        folium.CircleMarker(
            location=[row.poi_lat, row.poi_long],  # pyright: ignore[reportArgumentType]
            popup=f"POI: {row.poi_type}",
            color=color_map[row.poi_type],
            radius=5,
        ).add_to(m)
        folium.PolyLine(
            [(row.poi_lat, row.poi_long), (end_node.lat, end_node.long)],
            color=color_map[row.poi_type],
            weight=2,
            popup=f"POI: {row.poi_type}",
        ).add_to(m)

    paths = mcr_py.mcr.path.reconstruct_and_translate_path_for_label(
        path_manager.paths, label, translator_map
    )
    for i, path in enumerate(paths):
        if isinstance(path, Path):
            if path.path == []:
                continue
            if path.path_type == PathType.WALKING:
                walking_path_nodes = [nodes_by_id.loc[node_id] for node_id in path.path]
                path_lat_lon = [(node.lat, node.long) for node in walking_path_nodes]

                previous_meta = paths[i - 1].meta if i > 0 else None
                meta = format_meta(path.meta, previous_meta, start_time)
                if path_lat_lon != []:
                    folium.PolyLine(
                        [*path_lat_lon, (end_node.lat, end_node.long)],
                        color="grey",
                        weight=2,
                        popup=str(meta),
                    ).add_to(m)
            if path.path_type == PathType.CYCLING_WALKING:
                walking_path_nodes = [
                    nodes_by_id.loc[node_id]
                    for node_id in path.path
                    if isinstance(node_id, int)
                ]
                cycling_path_nodes = [
                    nodes_by_id.loc[int(node_id[1:])]
                    for node_id in path.path
                    if node_id[0] == "D"
                ]
                path_lat_lon = [(node.lat, node.long) for node in cycling_path_nodes]
                previous_meta = paths[i - 1].meta if i > 0 else None
                meta = format_meta(path.meta, previous_meta, start_time)
                if path_lat_lon != []:
                    folium.PolyLine(
                        path_lat_lon, color="blue", weight=2, popup=str(meta), dash_array=10
                    ).add_to(m)
        elif isinstance(path, GTFSPath):
            print("Impossible")
            start_stop_id = path.start_stop_id
            end_stop_id = path.end_stop_id
            start_stop = stops_by_id.loc[start_stop_id]
            end_stop = stops_by_id.loc[end_stop_id]
            trip = path.trip_id
            if len(trip) >= 10:
                trip = trip[:10] + "..."

            previous_meta = paths[i - 1].meta if i > 0 else None
            line_msg = f"Trip: {trip}\n---\n {format_meta(path.meta, previous_meta)}"

            path_lat_lon = [
                (float(start_stop.stop_lat), float(start_stop.stop_lon)),
                (float(end_stop.stop_lat), float(end_stop.stop_lon)),
            ]
            folium.PolyLine(
                path_lat_lon,
                color="green",
                weight=2,
                popup=line_msg,
            ).add_to(m)

            folium.CircleMarker(
                location=[float(start_stop.stop_lat), float(start_stop.stop_lon)],
                popup=f"Start: {start_stop.stop_name}",
                color="green",
                radius=3,
            ).add_to(m)
            folium.CircleMarker(
                location=[float(end_stop.stop_lat), float(end_stop.stop_lon)],
                popup=f"End: {end_stop.stop_name}",
                color="green",
                radius=3,
            ).add_to(m)
        else:
            raise Exception("Unknown path type")

folium.Map(location=[34.0522, -118.2437], zoom_start=12)

m